[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_22/2_sql.ipynb)

# Day 22: SQL (hard)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (26 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Revenue by country and year, side by side | hard | 4 |
| Q2 | Monthly retention matrix for a real shop (review) | hard | 7 |
| Q3 | Unpivot a wide dashboard table | hard | 5 |
| Q4 | Genre share of ratings by year | hard | 5 |
| Q5 | This year versus last year, per category | hard | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Real tables: Chinook** (a digital music store, 2009 to 2013; MIT license, lerocha/chinook-database):

| Table | One row is | Key columns |
|---|---|---|
| `Customer` | one customer (59) | `CustomerId`, `FirstName`, `LastName`, `Company` (often NULL), `City`, `State` (often NULL), `Country`, `Email`, `SupportRepId` |
| `Employee` | one employee (8) | `EmployeeId`, `FirstName`, `LastName`, `Title`, `ReportsTo` (manager id, NULL for the boss), `HireDate`, `City` |
| `Invoice` | one purchase (412) | `InvoiceId`, `CustomerId`, `InvoiceDate`, `BillingCity`, `BillingCountry`, `Total` |
| `InvoiceLine` | one track on an invoice (2,240) | `InvoiceLineId`, `InvoiceId`, `TrackId`, `UnitPrice`, `Quantity` |
| `Track` | one song or video (3,503) | `TrackId`, `Name`, `AlbumId`, `MediaTypeId`, `GenreId`, `Composer` (can be NULL), `Milliseconds`, `Bytes`, `UnitPrice` |
| `Album` / `Artist` | one album (347) / artist (275) | `AlbumId`, `Title`, `ArtistId` / `ArtistId`, `Name` |
| `Genre` / `MediaType` | one genre (25) / format (5) | `GenreId`, `Name` / `MediaTypeId`, `Name` |
| `Playlist` / `PlaylistTrack` | one playlist (18) / one track in a playlist (8,715) | `PlaylistId`, `Name` / `PlaylistId`, `TrackId` |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: Online Retail** (UCI, a UK online gift shop, 2010-12-01 to 2011-12-09, CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (text; a 'C' prefix = cancellation), `stock_code`, `description`, `quantity` (negative for cancellations), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price` (GBP), `customer_id` (NULL for about 25% of rows), `country` |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5.0 in steps of 0.5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title`, `year` (from the title, can be NULL), `genres` (pipe separated, e.g. 'Action|Comedy') |
| `ml_movie_genres` | one (movie, genre) pair | `movie_id`, `genre` (the `genres` string split into rows) |

**Real tables: Northwind** (a food wholesaler; orders 2012-07 to 2023-10; MIT license, jpwhite3/northwind-SQLite3):

| Table | One row is | Columns |
|---|---|---|
| `nw_orders` | one order (16,282) | `order_id`, `customer_id`, `employee_id`, `order_date` ('YYYY-MM-DD HH:MM:SS'), `shipped_date`, `ship_country` |
| `nw_order_items` | one product line of an order (609,283) | `order_id`, `product_id`, `unit_price`, `quantity`, `discount` (0 to 0.25). Revenue = `unit_price * quantity * (1 - discount)` |
| `nw_products` | one product (77) | `product_id`, `product`, `category_id`, `list_price`, `discontinued` |
| `nw_categories` | one category (8) | `category_id`, `category` |
| `nw_customers` | one customer (93) | `customer_id`, `company`, `city`, `country` |
| `nw_employees` | one employee (9) | `employee_id`, `name`, `title`, `reports_to` (manager, NULL for the boss) |

**Made-up wide table:** `platform_dau`: one row per day (2024-01-01 to 2024-03-31), columns `day`, `ios`,
`android`, `web` = daily active users per platform (a dashboard export).

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Real data: the Chinook digital music store (copied into the in-memory db).
_src = sqlite3.connect(data_path("chinook.sqlite", "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite"))
_src.backup(db)
_src.close()

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: UCI Online Retail (a UK gift shop, 2010-12 to 2011-12, CC BY 4.0).
import io, zipfile
_p = os.path.join(DATA, "online_retail.csv.gz")
if not os.path.exists(_p):                       # first run in Colab: download the UCI zip and convert it once
    with zipfile.ZipFile(data_path("online_retail.zip", "https://archive.ics.uci.edu/static/public/352/online+retail.zip")) as zf:
        pd.read_excel(io.BytesIO(zf.read("Online Retail.xlsx"))).to_csv(_p, index=False, compression="gzip")
_o = pd.read_csv(_p, dtype={"InvoiceNo": str, "StockCode": str})
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id", "country"]
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o.to_sql("retail", db, index=False)

# Real data: MovieLens small (GroupLens, research and education use). Downloads the zip if the CSVs are missing.
import zipfile
if not os.path.exists(os.path.join(DATA, "movielens_ratings.csv")):
    _z = data_path("ml-latest-small.zip", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip")
    with zipfile.ZipFile(_z) as zf:
        for _n in ("ratings", "movies"):
            with open(os.path.join(DATA, f"movielens_{_n}.csv"), "wb") as fh:
                fh.write(zf.read(f"ml-latest-small/{_n}.csv"))
_r = pd.read_csv(data_path("movielens_ratings.csv"))
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r = _r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]]
_r.to_sql("ml_ratings", db, index=False)
_m = pd.read_csv(data_path("movielens_movies.csv")).rename(columns={"movieId": "movie_id"})
_m["year"] = pd.to_numeric(_m["title"].str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce").astype("Int64")
_m[["movie_id", "title", "year", "genres"]].to_sql("ml_movies", db, index=False)
_g = _m.assign(genre=_m["genres"].str.split("|")).explode("genre")[["movie_id", "genre"]]
_g[_g["genre"] != "(no genres listed)"].to_sql("ml_movie_genres", db, index=False)

# Real data: Northwind orders (jpwhite3/northwind-SQLite3, MIT). Copied into db with short table names.
db.execute("ATTACH DATABASE ? AS nw", (data_path("northwind.db", "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"),))
db.executescript("""
CREATE TABLE nw_customers  AS SELECT CustomerID AS customer_id, CompanyName AS company, City AS city, Country AS country FROM nw.Customers;
CREATE TABLE nw_employees  AS SELECT EmployeeID AS employee_id, FirstName || ' ' || LastName AS name, Title AS title, ReportsTo AS reports_to FROM nw.Employees;
CREATE TABLE nw_categories AS SELECT CategoryID AS category_id, CategoryName AS category FROM nw.Categories;
CREATE TABLE nw_products   AS SELECT ProductID AS product_id, ProductName AS product, CategoryID AS category_id, UnitPrice AS list_price, Discontinued AS discontinued FROM nw.Products;
CREATE TABLE nw_orders     AS SELECT OrderID AS order_id, CustomerID AS customer_id, EmployeeID AS employee_id, OrderDate AS order_date, ShippedDate AS shipped_date, ShipCountry AS ship_country FROM nw.Orders;
CREATE TABLE nw_order_items AS SELECT OrderID AS order_id, ProductID AS product_id, UnitPrice AS unit_price, Quantity AS quantity, Discount AS discount FROM nw."Order Details";
""")
db.execute("DETACH DATABASE nw")

# Made-up dashboard export in WIDE format (one column per platform), built from the app tables above.
db.executescript("""
CREATE TABLE platform_dau AS
SELECT d.activity_date AS day,
  SUM(CASE WHEN u.platform = 'ios' THEN 1 ELSE 0 END) AS ios,
  SUM(CASE WHEN u.platform = 'android' THEN 1 ELSE 0 END) AS android,
  SUM(CASE WHEN u.platform = 'web' THEN 1 ELSE 0 END) AS web
FROM daily_activity d JOIN app_users u ON u.user_id = d.user_id
GROUP BY d.activity_date
ORDER BY d.activity_date;
""")

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Revenue by country and year, side by side

*hard, about 4 min*

Chinook (real). Finance wants one row per billing country and one column per year: `y2009` ... `y2013`
(revenue = sum of `Invoice.Total`, 2 decimals) plus `total`. Show only the 6 countries with the highest total,
highest first.

Example row shape: `USA | 103.95 | 102.98 | ... | 523.06`.

*Follow-up:* the years are not known in advance (new data arrives every year). What do you do?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'one column per year'. Pattern: pivot with conditional aggregation: `SUM(CASE WHEN year = ... THEN x ELSE 0 END)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. GROUP BY BillingCountry.
2. One `SUM(CASE WHEN strftime('%Y', InvoiceDate) = '2009' THEN Total ELSE 0 END)` per year.
3. SUM(Total) AS total; ORDER BY total DESC LIMIT 6.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT BillingCountry AS country,
  ROUND(SUM(CASE WHEN strftime('%Y', InvoiceDate) = '2009' THEN Total ELSE 0 END), 2) AS y2009,
  ROUND(SUM(CASE WHEN strftime('%Y', InvoiceDate) = '2010' THEN Total ELSE 0 END), 2) AS y2010,
  ROUND(SUM(CASE WHEN strftime('%Y', InvoiceDate) = '2011' THEN Total ELSE 0 END), 2) AS y2011,
  ROUND(SUM(CASE WHEN strftime('%Y', InvoiceDate) = '2012' THEN Total ELSE 0 END), 2) AS y2012,
  ROUND(SUM(CASE WHEN strftime('%Y', InvoiceDate) = '2013' THEN Total ELSE 0 END), 2) AS y2013,
  ROUND(SUM(Total), 2) AS total
FROM Invoice
GROUP BY BillingCountry
ORDER BY total DESC
LIMIT 6
```

Result:

| country | y2009 | y2010 | y2011 | y2012 | y2013 | total |
|---|---|---|---|---|---|---|
| USA | 103.95 | 102.98 | 103.01 | 127.98 | 85.14 | 523.06 |
| Canada | 57.42 | 76.26 | 55.44 | 42.57 | 72.27 | 303.96 |
| France | 35.64 | 39.6 | 42.61 | 36.66 | 40.59 | 195.1 |
| Brazil | 37.62 | 41.6 | 19.8 | 53.46 | 37.62 | 190.1 |
| Germany | 53.46 | 25.74 | 48.57 | 18.81 | 9.9 | 156.48 |
| United Kingdom | 25.74 | 30.69 | 17.82 | 9.9 | 28.71 | 112.86 |

**Why:** Every invoice row goes into exactly one CASE bucket, so the year columns add up to `total` (USA: 523.06). `ELSE 0` makes an empty year show 0 instead of NULL. Follow-up: SQL needs the column list at write time. Options: generate the query text in Python from `SELECT DISTINCT year`; PostgreSQL `crosstab` (tablefunc extension); or keep the result LONG (country, year, revenue) and pivot in pandas or the BI tool. In an exam, say the long format is the better storage format and the pivot is a presentation step.

**Complexity:** One scan and one GROUP BY: O(n).

**Common mistakes:** `COUNT(CASE ... THEN Total END)` (counts, not sums). Forgetting ELSE 0 (NULL cells). Filtering a year in WHERE (drops the other columns). PostgreSQL: `SUM(Total) FILTER (WHERE EXTRACT(year FROM InvoiceDate) = 2009)`.

**Learn more:** [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when)

</details>

---
### Q2. Monthly retention matrix for a real shop (review)

*hard, about 7 min*

UCI Online Retail (real). A customer's **cohort** is the month of their first purchase. For each cohort
return `customers` and `m1`, `m2`, `m3`: the % of the cohort that bought again 1, 2 and 3 calendar months later
(1 decimal). Only real purchases: `customer_id` not NULL, `invoice_no` not starting with 'C', `quantity > 0`.
A cell that cannot be observed yet (the month is after the last month in the data) must be **NULL, not 0**.

Example: first purchase in 2011-03; m1 = bought in 2011-04, m2 = 2011-05.

*Follow-up:* the data ends on 2011-12-09. Which cells are biased even though they are not NULL?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'cohort' and 'k months later' as columns. Pattern: retention plus pivot: month index difference, then `COUNT(DISTINCT CASE WHEN k = 1 THEN customer_id END)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `orders`: DISTINCT customer, month text, month index `year * 12 + month`.
2. `firsts`: MIN(month), MIN(index) per customer.
3. Join back: k = index - first index.
4. Per cohort: COUNT DISTINCT for k = 1, 2, 3 over cohort size.
5. Wrap each in `CASE WHEN first_index + k <= last_index THEN ... END` (last_index = MAX over all orders).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH orders AS (
  SELECT DISTINCT customer_id, substr(invoice_date, 1, 7) AS month,
    CAST(substr(invoice_date, 1, 4) AS INTEGER) * 12
      + CAST(substr(invoice_date, 6, 2) AS INTEGER) AS m
  FROM retail
  WHERE customer_id IS NOT NULL AND invoice_no NOT LIKE 'C%' AND quantity > 0
), firsts AS (
  SELECT customer_id, MIN(month) AS cohort, MIN(m) AS c
  FROM orders GROUP BY customer_id
), j AS (
  SELECT f.cohort, f.c, f.customer_id, o.m - f.c AS k
  FROM firsts f JOIN orders o ON o.customer_id = f.customer_id
), last AS (
  SELECT MAX(m) AS last_m FROM orders
)
SELECT cohort,
  COUNT(DISTINCT customer_id) AS customers,
  CASE WHEN c + 1 <= last_m THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN k = 1 THEN customer_id END)
       / COUNT(DISTINCT customer_id), 1) END AS m1,
  CASE WHEN c + 2 <= last_m THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN k = 2 THEN customer_id END)
       / COUNT(DISTINCT customer_id), 1) END AS m2,
  CASE WHEN c + 3 <= last_m THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN k = 3 THEN customer_id END)
       / COUNT(DISTINCT customer_id), 1) END AS m3
FROM j CROSS JOIN last
GROUP BY cohort, c, last_m
ORDER BY cohort
```

Result:

| cohort | customers | m1 | m2 | m3 |
|---|---|---|---|---|
| 2010-12 | 885 | 36.6 | 32.3 | 38.4 |
| 2011-01 | 417 | 22.1 | 26.6 | 23.0 |
| 2011-02 | 380 | 18.7 | 18.7 | 28.4 |
| 2011-03 | 452 | 15.0 | 25.2 | 19.9 |
| 2011-04 | 300 | 21.3 | 20.3 | 21.0 |
| 2011-05 | 284 | 19.0 | 17.3 | 17.3 |
| 2011-06 | 242 | 17.4 | 15.7 | 26.4 |
| 2011-07 | 188 | 18.1 | 20.7 | 22.3 |
| 2011-08 | 169 | 20.7 | 24.9 | 24.3 |
| 2011-09 | 299 | 23.4 | 30.1 | 11.4 |
| 2011-10 | 358 | 24.0 | 11.5 | NULL |
| 2011-11 | 324 | 11.1 | NULL | NULL |

(13 rows, first 12 shown)

**Why:** A month index (`year * 12 + month`) makes 'k months later' a subtraction that works across the year boundary (2010-12 to 2011-01 is k = 1). The k = 0 rows keep every customer in the cohort size. The CASE on `c + k <= last_m` separates 'nobody came back' (0) from 'we cannot know yet' (NULL): 2011-10 has m3 NULL, 2011-11 has m2 and m3 NULL. The first cohort (2010-12) is much stickier (m1 36.6%) because it holds all the loyal customers who bought before the data starts. Follow-up: every cell that lands in 2011-12 only has 9 days of data (2011-09 m3 = 11.4%, 2011-10 m2 = 11.5%, 2011-11 m1 = 11.1%), so they look low for no real reason.

**Complexity:** A DISTINCT and a self-join on customer_id: O(n log n).

**Common mistakes:** Subtracting month numbers without the year (December to January gives -11). Showing 0 for future months. Keeping cancellations ('C' invoices) as purchases. Using the first INVOICE line instead of the first month (a customer with two invoices in month 0 counts once, which DISTINCT on month handles).

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q3. Unpivot a wide dashboard table

*hard, about 5 min*

`platform_dau` (made up) is wide: `day | ios | android | web`. Turn it into a long table
`(day, platform, dau)` and use it to find, for each platform, its **peak day** (highest DAU; ties: earliest day), the
DAU on that day and `share_pct`: that platform's share of the day's total DAU across all three platforms (1 decimal).
Order by `dau` descending.

Example: on 2024-01-01 the wide row `12 | 12 | 6` becomes three long rows; ios share that day = 12 / 30 = 40.0%.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: columns that are really values of one variable. Pattern: unpivot with UNION ALL, then window functions (a ranking per platform and a SUM per day).

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `long`: three SELECTs (`'ios' AS platform, ios AS dau`), glued with UNION ALL.
2. In one CTE: `SUM(dau) OVER (PARTITION BY day)` and `ROW_NUMBER() OVER (PARTITION BY platform ORDER BY dau DESC, day)`.
3. Keep rn = 1.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH long AS (
  SELECT day, 'ios' AS platform, ios AS dau FROM platform_dau
  UNION ALL
  SELECT day, 'android', android FROM platform_dau
  UNION ALL
  SELECT day, 'web', web FROM platform_dau
), ranked AS (
  SELECT day, platform, dau,
    SUM(dau) OVER (PARTITION BY day) AS day_total,
    ROW_NUMBER() OVER (PARTITION BY platform ORDER BY dau DESC, day) AS rn
  FROM long
)
SELECT platform, day AS peak_day, dau, ROUND(100.0 * dau / day_total, 1) AS share_pct
FROM ranked
WHERE rn = 1
ORDER BY dau DESC
```

Result:

| platform | peak_day | dau | share_pct |
|---|---|---|---|
| android | 2024-02-24 | 147 | 51.8 |
| ios | 2024-02-19 | 127 | 44.1 |
| web | 2024-02-05 | 40 | 16.6 |

**Why:** Once the data is long, 'per platform' is just PARTITION BY platform, and adding a fourth platform needs no query change. The day total is computed in the same CTE as the rank, BEFORE the `rn = 1` filter; computed after the filter it would only see one row per day. Android peaks on 2024-02-24 with 147 users (51.8% of that day).

**Complexity:** O(n) to unpivot (3 rows per day) plus window sorts.

**Common mistakes:** UNION instead of UNION ALL (removes rows that happen to be equal). Taking the share after filtering rn = 1. PostgreSQL has a shorter form: `CROSS JOIN LATERAL (VALUES ('ios', ios), ('android', android), ('web', web)) v(platform, dau)`.

**Learn more:** [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-window-ranking](https://mahsa-agz.github.io/ds-handbook/#sql-window-ranking)

</details>

---
### Q4. Genre share of ratings by year

*hard, about 5 min*

MovieLens (real). For the years 2015 to 2018 (by `rated_at`), build a table with one row per genre and
columns `p2015`, `p2016`, `p2017`, `p2018`: the % of **all ratings of that year** that went to a movie of that genre
(1 decimal). Show the 6 genres with the highest `p2018`.

Example: 2018 has 6,418 ratings; if 2,000 of them are for Comedy movies, Comedy's p2018 = 31.2.

*Follow-up:* the column for 2018 adds up to far more than 100%. Bug or not?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: rows = genre, columns = year, cell = share of the column total. Pattern: pivot with conditional aggregation, divided by a per-year denominator.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE `r`: ratings joined to `ml_movie_genres`, with `yr = strftime('%Y', rated_at)`, from 2015 on.
2. CTE `tot`: ratings per year from `ml_ratings` (NOT from the join).
3. Per genre: `100.0 * SUM(CASE WHEN yr = '2018' THEN 1 ELSE 0 END) / (SELECT n FROM tot WHERE yr = '2018')`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH r AS (
  SELECT g.genre, strftime('%Y', r.rated_at) AS yr
  FROM ml_ratings r
  JOIN ml_movie_genres g ON g.movie_id = r.movie_id
  WHERE r.rated_at >= '2015-01-01'
), tot AS (
  SELECT strftime('%Y', rated_at) AS yr, COUNT(*) AS n
  FROM ml_ratings
  WHERE rated_at >= '2015-01-01'
  GROUP BY strftime('%Y', rated_at)
)
SELECT genre,
  ROUND(100.0 * SUM(CASE WHEN yr = '2015' THEN 1 ELSE 0 END) / (SELECT n FROM tot WHERE yr = '2015'), 1) AS p2015,
  ROUND(100.0 * SUM(CASE WHEN yr = '2016' THEN 1 ELSE 0 END) / (SELECT n FROM tot WHERE yr = '2016'), 1) AS p2016,
  ROUND(100.0 * SUM(CASE WHEN yr = '2017' THEN 1 ELSE 0 END) / (SELECT n FROM tot WHERE yr = '2017'), 1) AS p2017,
  ROUND(100.0 * SUM(CASE WHEN yr = '2018' THEN 1 ELSE 0 END) / (SELECT n FROM tot WHERE yr = '2018'), 1) AS p2018
FROM r
GROUP BY genre
ORDER BY p2018 DESC
LIMIT 6
```

Result:

| genre | p2015 | p2016 | p2017 | p2018 |
|---|---|---|---|---|
| Comedy | 38.3 | 38.0 | 34.7 | 40.4 |
| Drama | 44.0 | 40.6 | 37.9 | 34.2 |
| Action | 30.3 | 36.5 | 37.8 | 32.2 |
| Adventure | 26.8 | 28.0 | 28.1 | 29.0 |
| Sci-Fi | 18.0 | 20.8 | 21.8 | 20.4 |
| Thriller | 24.4 | 27.1 | 28.9 | 19.9 |

**Why:** The denominator must be the number of RATINGS per year, counted before the genre join; counting after the join would count a 3 genre movie three times. Follow-up: not a bug. A movie has several genres, so a rating is counted once in every genre column; the shares are 'percent of ratings that touch this genre', and they can sum above 100. Comedy leads 2018 with 40.4% while Drama falls from 44.0% (2015) to 34.2% (2018).

**Complexity:** Join of ratings to genres (about 2.2 rows per rating) and one GROUP BY.

**Common mistakes:** Dividing by the joined row count. Integer division. Filtering `yr = '2018'` in WHERE, which empties the other columns.

**Learn more:** [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q5. This year versus last year, per category

*hard, about 5 min*

Northwind (real). Revenue = `unit_price * quantity * (1 - discount)`. The data stops on 2023-10-28, so
compare the same window in both years: **January to September**. Return per category `rev_2022`, `rev_2023`
(rounded to whole numbers) and `growth_pct = (rev_2023 / rev_2022 - 1) * 100` (1 decimal), ordered by growth descending.

*Follow-up:* why not compare full-year 2022 with all of 2023?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: two periods as two columns, then a ratio. Pattern: pivot by period with conditional sums, then compute growth in an outer query (or repeat the expressions).

</details>

<details><summary><b>Hint 2</b></summary>

1. Join orders, items, products, categories; WHERE order_date in 2022 or 2023 (Jan to Sep).
2. `SUM(CASE WHEN order_date >= '2022-01-01' AND order_date < '2022-10-01' THEN rev ELSE 0 END)` and the same for 2023.
3. Outer SELECT: growth from the two columns.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH lines AS (
  SELECT c.category, o.order_date,
    i.unit_price * i.quantity * (1 - i.discount) AS revenue
  FROM nw_orders o
  JOIN nw_order_items i ON i.order_id = o.order_id
  JOIN nw_products p ON p.product_id = i.product_id
  JOIN nw_categories c ON c.category_id = p.category_id
  WHERE (o.order_date >= '2022-01-01' AND o.order_date < '2022-10-01')
     OR (o.order_date >= '2023-01-01' AND o.order_date < '2023-10-01')
), by_cat AS (
  SELECT category,
    SUM(CASE WHEN order_date < '2023-01-01' THEN revenue ELSE 0 END) AS rev_2022,
    SUM(CASE WHEN order_date >= '2023-01-01' THEN revenue ELSE 0 END) AS rev_2023
  FROM lines
  GROUP BY category
)
SELECT category, ROUND(rev_2022) AS rev_2022, ROUND(rev_2023) AS rev_2023,
  ROUND((rev_2023 / rev_2022 - 1) * 100, 1) AS growth_pct
FROM by_cat
ORDER BY growth_pct DESC
```

Result:

| category | rev_2022 | rev_2023 | growth_pct |
|---|---|---|---|
| Seafood | 3210027.0 | 3416972.0 | 6.4 |
| Produce | 2090832.0 | 2205406.0 | 5.5 |
| Condiments | 3621229.0 | 3807802.0 | 5.2 |
| Meat/Poultry | 4101412.0 | 4315451.0 | 5.2 |
| Grains/Cereals | 1816849.0 | 1907634.0 | 5.0 |
| Confections | 4322090.0 | 4503797.0 | 4.2 |
| Dairy Products | 3772993.0 | 3848592.0 | 2.0 |
| Beverages | 6060877.0 | 6125471.0 | 1.1 |

**Why:** Filtering both windows in WHERE first keeps the scan small; the CASE then splits the rows into the two columns. Growth is computed from the UNROUNDED sums in the outer query. Every category grows 1% to 6%; Seafood grows most. Follow-up: 2023 is incomplete (no data after 2023-10-28), so a full-year comparison shows a fake drop of about 17% (39.7 million versus 33.1 million); like-for-like windows (or year to date) are the rule.

**Complexity:** One join over about 600k item lines, filtered by date first.

**Common mistakes:** Comparing a partial year with a full year. Growth from rounded values. Forgetting the discount. Dividing by a zero baseline for a new category (guard with NULLIF).

**Learn more:** [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_22/3_stats.ipynb)